# Public Trade Analysis

Phase 1: load raw JSON/JSONL trades, preprocess, and save to `preprocessed_trades.parquet`.

In [ ]:
import datetime
from pathlib import Path

from polymarket_analysis.preprocessing.loader import (
    day_folders,
    load_all_markets_and_trades,
)
from polymarket_analysis.preprocessing.trades import (
    build_raw_dataframe,
    aggregate_trades,
    compute_wallet_summary,
    filter_top_wallets,
)


In [ ]:
# ── configuration ─────────────────────────────────────────────────────────────
START_DATE     = datetime.date(2025, 1, 1)
END_DATE       = datetime.date(2026, 3, 15)

# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Wallet ranking is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 10)

raise NotImplementedError("Very slow, do not run.")

TRADES_RAW     = Path("../data/trades_raw")
TRADES_PARQUET = Path("../data/polygon_trades_processed/from_json.parquet")

TRADES_PARQUET.parent.mkdir(parents=True, exist_ok=True)


In [ ]:
# ── load raw data ─────────────────────────────────────────────────────────────
folders = day_folders(TRADES_RAW, START_DATE, END_DATE)
print(f"Day folders found: {len(folders)}")

markets, all_trades = load_all_markets_and_trades(folders)
print(f"Loaded {len(all_trades):,} trade records across {len(markets):,} markets.")


In [ ]:
# ── build dataframe and join token resolution data ─────────────────────────────
df = build_raw_dataframe(all_trades, markets)
df = aggregate_trades(df)
print(f"Aggregated rows: {len(df):,}")
df.head(3)


In [ ]:
# ── wallet summary and top-wallet selection ────────────────────────────────────
wallet_summary = compute_wallet_summary(df, end_date_train=END_DATE_TRAIN)
print(f"Unique wallets (training period): {len(wallet_summary):,}")
wallet_summary.describe()


In [ ]:
top_trades = filter_top_wallets(wallet_summary, df, end_date_train=END_DATE_TRAIN)
top_wallets = set(top_trades["wallet"].unique())
pnl_threshold = wallet_summary["pnl_usdc"].quantile(0.95)
print(f"PnL threshold (95th pct, training): {pnl_threshold:,.2f} USDC")
print(f"Top-5% wallet count: {len(top_wallets):,}")


In [ ]:
# ── save preprocessed trades ──────────────────────────────────────────────────
print(f"Total rows: {len(top_trades):,}  train={top_trades['is_train'].sum():,}  test={(~top_trades['is_train']).sum():,}")
top_trades.to_parquet(TRADES_PARQUET, index=False)
print(f"Saved → {TRADES_PARQUET}")
